# Catenary / energization LaTeX figures

Generates SVG figures for a LaTeX document from an energization sweep.

**Figures**
1. Corridor charging load curves (catenary + stationary), four versions:
   week / single-day Ã— broken-out-by-charging-type / combined.
1a. Single mid-route substation load curve (single day).
2. Lifecycle GHG intensity (kg CO2-eq / tonne-km), 3Ã—3 cost-scenario grid.
3. Lifecycle energy intensity (kWh-eq / tonne-km), stacked by source, 3Ã—3 grid.

**Notes / assumptions**
- The code is *data-driven*: it discovers scenarios (architecture, variant,
  infra-cost level, energy-price level) from the sweep folder, so it expands
  automatically as the still-running sweep adds architectures / a diesel case.
- Emissions: `GHG_Energy` [tonne CO2-eq] summed over rollout years, divided by
  `Freight_Moved` [million tonne-km] summed over years.
- Energy: `Energy` [MJ] by `Subset`, converted MJâ†’kWh (Ã·3.6). Grouping:
  *Liquid* = `Diesel`; *Catenary* = `Electricity` + `Electricity_Line_Loss`;
  *Stationary charging* = `Electricity_Stationary_Charging`.
- Load curves: catenary grid draw = summed `substation_load_curves.parquet`,
  split into *Delivered* vs *Losses* using the case's line-loss fraction from
  the metrics; stationary draw = constant `peak_kw` between `start_hr`/`end_hr`
  from `stationary_charging.parquet` (absent until the run schedules sessions).


In [ ]:
from pathlib import Path

import polars as pl
import altair as alt

alt.data_transformers.disable_max_rows()

SWEEP_DIR = Path(
    r"C:\Users\mbruchon\Documents\Repos\NREL\altrios-applied\applications"
    r"\catenary_optimization\outputs\energization_results\sweep_20260801_001256"
)
FIG_DIR = SWEEP_DIR / "latex_figures"
FIG_DIR.mkdir(exist_ok=True)

MJ_PER_KWH = 3.6
LEVEL_ORDER = ["low", "med", "high"]  # cost-scenario facet order
LEVEL_LABELS = {"low": "Low", "med": "Central", "high": "High"}  # facet display text
# Vega expression mapping a level id -> its display label (for axis/facet headers).
LEVEL_LABEL_EXPR = "{'low': 'Low', 'med': 'Central', 'high': 'High'}[datum.value]"

# Report only the base demand level. (The sweep's diesel + freight aren't demand-
# scaled, so the high-demand cases can't be trusted; restrict everything to base.)
REPORT_DEMAND = "central"

# Scenario used for the load-curve figures (plots 1 & 1a).
LOADCURVE_VARIANT = "bel"
LOADCURVE_INFRA = "low"
LOADCURVE_ENERGY = "med"
LOADCURVE_DEMAND = "central"
LOADCURVE_DAY_INDEX = 3  # simulation day shown in the single-day figures


def save_svg(chart, name):
    path = FIG_DIR / f"{name}.svg"
    chart.save(str(path))

    print("saved", path.relative_to(SWEEP_DIR.parent))
    return chart

## Discover scenarios and load metrics

In [ ]:
def scenario_registry(sweep_dir):
    # One row per run folder with its scenario dimensions + folder path.
    rows = []
    for folder in sorted(p for p in sweep_dir.iterdir()
                         if p.is_dir() and not p.name.startswith("_")):
        mp = folder / "metrics.parquet"
        if not mp.exists():
            continue
        meta = pl.read_parquet(mp).select(
            "architecture", "variant", "loco_type",
            "demand_level", "infra_level", "energy_level").head(1)
        if meta.height:
            d = meta.to_dicts()[0]
            d["folder"] = str(folder)
            rows.append(d)
    return pl.DataFrame(rows)


# Base demand only (see REPORT_DEMAND).
registry = scenario_registry(SWEEP_DIR).filter(pl.col("demand_level") == REPORT_DEMAND)
registry


In [ ]:
metrics = (
    pl.read_csv(SWEEP_DIR / "energization_metrics_all.csv", infer_schema_length=0)
    .with_columns(pl.col("Value").cast(pl.Float64, strict=False))
    # Base demand only (see REPORT_DEMAND); high-demand diesel/freight aren't scaled.
    .filter(pl.col("demand_level") == REPORT_DEMAND)
)

CASE_KEYS = ["architecture", "demand_level", "infra_level",
             "energy_level", "variant", "loco_type"]

# The all-diesel baseline is architecture- AND infrastructure-independent, so its
# rows carry architecture="none" / infra_level="na" (only energy_level varies).
# Replicate them into every reported infra level so the baseline appears in each
# infra facet row/cell of Figs 2-5. No-op until the diesel run lands its "na" rows.
INFRA_LEVELS_REPORT = ["low", "med", "high"]


def broadcast_infra_independent(df):
    na = df.filter(pl.col("infra_level") == "na")
    if na.height == 0:
        return df
    expanded = pl.concat(
        [na.with_columns(pl.lit(lvl).alias("infra_level"))
         for lvl in INFRA_LEVELS_REPORT], how="diagonal_relaxed")
    return pl.concat([df.filter(pl.col("infra_level") != "na"), expanded],
                     how="diagonal_relaxed")


metrics = broadcast_infra_independent(metrics)

# The sweep sim represents one WEEK, but the metric calculator annualized diesel,
# freight, and distances via the Rust `annualize` flag (x365.25) while electricity
# used 365.25/PLAN_DAYS (week basis). Those Rust-annualized quantities are therefore
# PLAN_DAYS (=7) too high; divide them back onto the week basis wherever used below
# (freight here; diesel energy/GHG/cost in Figs 2-4). Verified: factor is exactly 7.
PLAN_DAYS = 7.0


def add_case_labels(df):
    family = (
        pl.when(pl.col("variant") == "bel").then(pl.lit("BEL"))
        .when(pl.col("variant") == "dual_mode").then(pl.lit("Dual-Mode"))
        .when(pl.col("variant") == "diesel").then(pl.lit("Diesel"))
        .otherwise(pl.col("variant"))
    )
    return df.with_columns(family.alias("tech_family")).with_columns(
        pl.when(pl.col("variant") == "diesel")
        .then(pl.col("tech_family"))
        .otherwise(pl.col("tech_family") + pl.lit(" \u00b7 ") + pl.col("architecture"))
        .alias("tech_option")
    )


# Freight (million tonne-km) summed over rollout years, per case (shared by 2 & 3).
# /PLAN_DAYS corrects the Rust over-annualization so it sits on the week basis.
freight = (
    metrics.filter((pl.col("Metric") == "Freight_Moved")
                   & (pl.col("Units") == "million tonne-km"))
    .group_by(CASE_KEYS).agg((pl.col("Value").sum() / PLAN_DAYS).alias("freight_mtkm"))
)
freight.head()


## Reconstruct stationary (terminal) charging from `train_timeseries.parquet`

The completed sweep's metrics excluded stationary charging (the serialization bug
was fixed after the run finished, and the sweep can't be re-run). Reconstruct it
here for the **BEL** variants: each BEL locomotive ends its run at some battery
SOC < 100%, and topping it back to full at the terminal draws
`9600 kWh Â· (1 âˆ’ SOC_end) / 0.9` from the grid (0.9 charger efficiency). Dispatch
times come from the consist plan, and the model's 750 kW delivered charger gives
each session a constant-power window (`start_hr`/`end_hr`) for load-curve plotting.
Outputs: `stationary_sessions` (one row per BEL loco per run) and
`stationary_by_case` (grid kWh / MJ summed per case).


In [ ]:
import re

# --- Reconstructed stationary (terminal) charging for BEL variants ---------------
# The completed sweep's metrics omitted stationary charging (the serialization bug
# was fixed only after the run). Recover it from each BEL locomotive's end-of-run
# battery SOC: recharging back to 100% at the terminal delivers
# BEL_BATTERY_KWH * (1 - soc_end) to the battery and draws that / efficiency from
# the grid. Charger matches the model (altrios.defaults): 750 kW delivered @ 0.9,
# so each session is a constant-power rectangle usable as a load curve.
BEL_BATTERY_KWH = 9600.0
BEL_CHARGER_EFF = 0.9
BEL_CHARGER_KW = 750.0  # delivered (battery-side); grid draw = BEL_CHARGER_KW / eff

CONSIST_PLAN_PATH = (
    SWEEP_DIR.parents[2] / "train_consist_plan_spokane_lethbridge_clean.csv")
# Each train's planned departure + the terminal it ends its run at (its
# Destination): the terminal fixes which grid region the terminal charging draws on.
departures = (
    pl.read_csv(CONSIST_PLAN_PATH)
    .group_by("Train_ID")
    .agg(pl.first("Departure_Time_Planned_Hr").alias("departure_hr"),
         pl.first("Destination_ID").alias("destination"))
    .with_columns(pl.col("Train_ID").cast(pl.Utf8).alias("train_id"))
    .select("train_id", "departure_hr", "destination")
)

_SOC_RE = re.compile(
    r"loco_con\.loco_vec\.\[(\d+)\]\.loco_type\.BatteryElectricLoco"
    r"\.res\.history\.soc$")


def bel_stationary_sessions(reg_row):
    # One row per (train, BEL loco): the terminal recharge to full after its run.
    lf = pl.scan_parquet(Path(reg_row["folder"]) / "train_timeseries.parquet")
    soc_cols = [c for c in lf.collect_schema().names() if _SOC_RE.match(c)]
    if not soc_cols:  # dual-mode / diesel folders have no BEL battery
        return None
    tt = lf.select("train_id", "train_index",
                   "history.time_seconds", *soc_cols).collect()
    # End-of-run row per train (max sim time = arrival at the terminal).
    last = (tt.sort("history.time_seconds")
              .group_by("train_index", maintain_order=True).last())
    sess = (
        last.unpivot(index=["train_id", "history.time_seconds"], on=soc_cols,
                     variable_name="soc_col", value_name="soc_end")
        .drop_nulls("soc_end")
        .with_columns(
            pl.col("soc_col").str.extract(r"loco_vec\.\[(\d+)\]", 1)
                .cast(pl.Int64).alias("loco_idx"),
            (BEL_BATTERY_KWH * (1.0 - pl.col("soc_end")).clip(lower_bound=0.0))
                .alias("delivered_kwh"),
            (pl.col("history.time_seconds") / 3600.0).alias("run_end_hr"))
        .with_columns(
            (pl.col("delivered_kwh") / BEL_CHARGER_EFF).alias("grid_kwh"),
            (pl.col("delivered_kwh") / BEL_CHARGER_KW).alias("duration_hr"),
            pl.lit(BEL_CHARGER_KW / BEL_CHARGER_EFF).alias("grid_kw"))
        .join(departures, on="train_id", how="left")
        .with_columns((pl.col("departure_hr") + pl.col("run_end_hr")).alias("start_hr"))
        .with_columns((pl.col("start_hr") + pl.col("duration_hr")).alias("end_hr"))
    )
    for k in CASE_KEYS:
        sess = sess.with_columns(pl.lit(reg_row[k]).alias(k))
    return sess.select(*CASE_KEYS, "train_id", "loco_idx", "destination", "soc_end",
                       "delivered_kwh", "grid_kwh", "grid_kw",
                       "start_hr", "end_hr", "duration_hr")


_parts = [s for s in (bel_stationary_sessions(r)
                      for r in registry.filter(pl.col("variant") == "bel").to_dicts())
          if s is not None and s.height]
stationary_sessions = (pl.concat(_parts, how="diagonal_relaxed") if _parts
                       else pl.DataFrame())

# Grid electricity from stationary charging, summed per case. MJ matches the
# metrics' Energy units so it can slot into the energy / emissions figures.
stationary_by_case = (
    stationary_sessions.group_by(CASE_KEYS)
    .agg(pl.col("grid_kwh").sum().alias("stationary_grid_kwh"),
         (pl.col("grid_kwh").sum() * MJ_PER_KWH).alias("stationary_grid_mj"),
         pl.len().alias("n_sessions"))
    .pipe(add_case_labels)
    .sort("tech_option")
    if stationary_sessions.height else pl.DataFrame()
)
print(f"reconstructed {stationary_sessions.height} BEL stationary sessions "
      f"across {stationary_by_case.height} cases")
stationary_by_case


### Discounted cost and emissions of stationary charging

Turn the reconstructed grid electricity into the cost and emissions terms the sweep
metrics missed, following `energization_optimization_corridor.py`. Cost has two
parts, both discounted at 7 %/yr over the 30-year rollout: **(a)** the AEO electricity
energy cost (energy-price multiplier `1 Â± 1/3`, where the "low" label is the low
fuel-price *differential* = the **dear**-electricity case), and **(b)** the terminal
**charger capital** â€” the sweep never modeled stationary charging, so the chargers
were never costed; size them by the peak concurrent sessions per terminal (one 750 kW
port each, `defaults.BEL_CHARGER_COST_USD`, replaced every `BEL_CHARGER_LIFESPAN`
years). Emissions use **two** Cambium regions by terminal (Spokane â†’
`NorthernGrid_West`, Lethbridge â†’ dirtier `NorthernGrid_East`) and include line +
charger losses. Produces `stationary_econ` (per case: energy + charger cost, lifetime
COâ‚‚, rollout energy, port count, and the LCOTKM increment) for Figures 2â€“4.


In [ ]:
import altrios.metric_calculator as mc
from altrios import defaults

# --- Cost + emissions of the reconstructed stationary charging --------------------
# Replicates energization_optimization_corridor.py. The plan's week of grid
# electricity is annualized (365.25 / PLAN_DAYS x demand multiplier) and rolled out
# over METRICS_ROLLOUT_YEARS identical operating years.
#   Cost: (a) AEO industrial electricity price x the scenario's energy-price
#     multiplier, discounted at DISCOUNT_RATE per year (region-uniform, so
#     region-independent); PLUS (b) terminal CHARGER CAPITAL -- the sweep never
#     modeled stationary charging, so the chargers were never costed either. Size
#     the charger fleet by the peak concurrent sessions per terminal (one 750 kW
#     port each), priced at defaults.BEL_CHARGER_COST_USD and replaced every
#     BEL_CHARGER_LIFESPAN years across the rollout.
#     NB: the energy-level label is the fuel-price DIFFERENTIAL, so "low" (barely
#     favors electrification) is the DEAR-electricity case (x 1 + 1/3) and "high"
#     is the cheap-electricity case (x 1 - 1/3) -- matching ENERGY_LEVELS.
#   Emissions: Cambium CO2e factor per year, attributed to each session's TERMINAL
#     grid region -- trains ending at Spokane draw NorthernGrid_West, those ending
#     at Lethbridge the (dirtier) NorthernGrid_East. Includes line + charger losses
#     (grid-side kWh).
GRID_REGION_WEST = "NorthernGrid_West"   # Spokane terminus
GRID_REGION_EAST = "NorthernGrid_East"   # Lethbridge terminus
TERMINUS_REGION = {"Spokane": GRID_REGION_WEST, "Lethbridge": GRID_REGION_EAST}
CAMBIUM_SCENARIO = "MidCase"
METRICS_START_YEAR = 2025
METRICS_ROLLOUT_YEARS = 30
DISCOUNT_RATE = 0.07
PLAN_DAYS = 7.0
ANNUAL_SCALER = 365.25 / PLAN_DAYS          # week -> year
DEMAND_MULT = {"central": 1.0, "high": 2.0}
_GAP = 1.0 / 3.0
ELEC_MULT = {"low": 1.0 + _GAP, "med": 1.0, "high": 1.0 - _GAP}
_ROLLOUT_YEARS = [METRICS_START_YEAR + k for k in range(METRICS_ROLLOUT_YEARS)]


def _aeo_elec_price(year):
    # AEO industrial electricity price at `year` in $/kWh (table is cents/kWh).
    yrs = mc.electricity_prices_eia.get_column("Year")
    y = min(max(int(year), int(yrs.min())), int(yrs.max()))
    return float(mc.electricity_prices_eia
                 .filter(pl.col("Year") == y).get_column("Price")[0]) / 100.0


# Summed CO2e factor (kg/MWh, undiscounted) per terminal region over the rollout.
_ef = mc.interpolate_emissions_factors_cambium(
    [GRID_REGION_WEST, GRID_REGION_EAST], _ROLLOUT_YEARS, CAMBIUM_SCENARIO)
_EF_SUM_BY_REGION = dict(
    _ef.group_by("Node").agg(pl.col("CO2eq_kg_per_MWh").sum()).iter_rows())

# Year-independent rollout aggregates: discounted electricity price PV ($/kWh), the
# annuity factor for the (per-year constant) freight, and the discounted charger
# replacement factor (buy at years 0, 15, ... within the rollout).
_PRICE_PV = sum(_aeo_elec_price(y) / (1.0 + DISCOUNT_RATE) ** k
                for k, y in enumerate(_ROLLOUT_YEARS))
_PVAF = sum(1.0 / (1.0 + DISCOUNT_RATE) ** k for k in range(METRICS_ROLLOUT_YEARS))
_CHARGER_PV = sum(1.0 / (1.0 + DISCOUNT_RATE) ** y
                  for y in range(0, METRICS_ROLLOUT_YEARS, defaults.BEL_CHARGER_LIFESPAN))

# Charger fleet size = peak concurrent sessions per terminal (one 750 kW port each),
# from a sweep line over the session start/end times; summed across both terminals.
_ev = pl.concat([
    stationary_sessions.select(*CASE_KEYS, pl.col("destination").alias("terminal"),
                               pl.col("start_hr").alias("t"), pl.lit(1).alias("d")),
    stationary_sessions.select(*CASE_KEYS, pl.col("destination").alias("terminal"),
                               pl.col("end_hr").alias("t"), pl.lit(-1).alias("d")),
])
ports = (
    _ev.sort(CASE_KEYS + ["terminal", "t", "d"])   # end (-1) before start (+1) at ties
    .with_columns(pl.col("d").cum_sum().over(CASE_KEYS + ["terminal"]).alias("conc"))
    .group_by(CASE_KEYS + ["terminal"]).agg(pl.col("conc").max().alias("peak"))
    .group_by(CASE_KEYS).agg(pl.col("peak").sum().alias("n_ports"))
)

# Emissions: split the week of charging by terminal region, annualize, and apply
# that region's Cambium factor; sum the two regions back to one value per case.
stat_ghg = (
    stationary_sessions
    .with_columns(pl.col("destination").replace_strict(
        TERMINUS_REGION, return_dtype=pl.Utf8).alias("region"))
    .group_by(CASE_KEYS + ["region"])
    .agg(pl.col("grid_kwh").sum().alias("region_week_kwh"))
    .with_columns(
        pl.col("demand_level").replace_strict(DEMAND_MULT, return_dtype=pl.Float64)
            .alias("_dmult"),
        pl.col("region").replace_strict(_EF_SUM_BY_REGION, return_dtype=pl.Float64)
            .alias("_ef_sum"))
    .with_columns(
        (pl.col("region_week_kwh") * ANNUAL_SCALER * pl.col("_dmult")
         * pl.col("_ef_sum") / 1e6).alias("region_ghg_tonne"))
    .group_by(CASE_KEYS)
    .agg(pl.col("region_ghg_tonne").sum().alias("stat_ghg_tonne"))
)

stationary_econ = (
    stationary_by_case.select(*CASE_KEYS, "stationary_grid_kwh")
    .with_columns(
        pl.col("demand_level").replace_strict(DEMAND_MULT, return_dtype=pl.Float64)
            .alias("_dmult"),
        pl.col("energy_level").replace_strict(ELEC_MULT, return_dtype=pl.Float64)
            .alias("_emult"))
    .join(ports, on=CASE_KEYS, how="left")
    .with_columns(
        (pl.col("stationary_grid_kwh") * ANNUAL_SCALER * pl.col("_dmult"))
            .alias("annual_stat_kwh"))
    .with_columns(
        (pl.col("annual_stat_kwh") * pl.col("_emult") * _PRICE_PV)
            .alias("stat_energy_cost_usd"),
        # More trains (demand) -> proportionally more ports; discounted replacements.
        (pl.col("n_ports") * pl.col("_dmult")
         * defaults.BEL_CHARGER_COST_USD * _CHARGER_PV).alias("stat_charger_cost_usd"),
        (pl.col("annual_stat_kwh") * METRICS_ROLLOUT_YEARS * MJ_PER_KWH)
            .alias("stat_energy_mj_total"))
    .with_columns(
        (pl.col("stat_energy_cost_usd") + pl.col("stat_charger_cost_usd"))
            .alias("stat_disc_cost_usd"))
    .join(stat_ghg, on=CASE_KEYS, how="left")
    .join(freight, on=CASE_KEYS, how="left")
    # LCOTKM delta = added discounted cost / discounted freight (annual x annuity).
    .with_columns(
        (pl.col("stat_disc_cost_usd")
         / ((pl.col("freight_mtkm") / METRICS_ROLLOUT_YEARS) * _PVAF))
            .alias("delta_lcotkm"))
    .drop("_dmult", "_emult")
)
stationary_econ.select(*CASE_KEYS, "n_ports", "stat_energy_cost_usd",
                       "stat_charger_cost_usd", "stat_ghg_tonne", "delta_lcotkm"
                       ).sort("delta_lcotkm")


In [ ]:
import colorsys

# ---- Consistent technology / architecture color scheme (shared by all figures) ----
# Standard D3 / Tableau category10 hues (matplotlib / Altair defaults): one base per
# technology; catenary architectures within a technology are drawn as lighter -> darker
# shades of that hue.
TECH_ORDER = ["BEL", "Dual-Mode", "Diesel"]
TECH_BASE_COLORS = {
    "BEL": "#2ca02c",        # green
    "Dual-Mode": "#1f77b4",  # blue
    "Diesel": "#7f7f7f",     # gray
}


def _shade(hex_color, factor):
    # factor > 0 lightens toward white, < 0 darkens toward black (factor in [-1, 1]).
    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = l + (1.0 - l) * factor if factor >= 0 else l * (1.0 + factor)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))


def tech_family_scale():
    # Color scale for the coarse technology family (one hue each).
    return alt.Scale(domain=TECH_ORDER, range=[TECH_BASE_COLORS[t] for t in TECH_ORDER])


def tech_option_scale(df):
    # Color scale for tech_option: each family's hue, split into lighter -> darker
    # shades across its architectures. Returns (scale, ordered domain).
    present = df.select("tech_family", "tech_option").unique()
    domain, rng = [], []
    for fam in TECH_ORDER:
        opts = sorted(present.filter(pl.col("tech_family") == fam)["tech_option"]
                      .unique().to_list())
        n = len(opts)
        for i, opt in enumerate(opts):
            factor = 0.0 if n == 1 else (1.0 - 2.0 * i / (n - 1)) * 0.45
            domain.append(opt)
            rng.append(_shade(TECH_BASE_COLORS[fam], factor))
    return alt.Scale(domain=domain, range=rng), domain


## Figure 2 â€” Lifecycle GHG intensity (kg CO2-eq / tonne-km)

In [ ]:
emissions = (
    # Rebuild GHG_Energy from components so the over-annualized diesel term can be
    # put on the week basis (/PLAN_DAYS); electricity is already correct.
    metrics.filter(pl.col("Metric").is_in(["GHG_Diesel", "GHG_Electricity"]))
    .with_columns(pl.when(pl.col("Metric") == "GHG_Diesel")
                    .then(pl.col("Value") / PLAN_DAYS)
                    .otherwise(pl.col("Value")).alias("Value"))
    .group_by(CASE_KEYS).agg(pl.col("Value").sum().alias("ghg_tonne"))
    # Add reconstructed stationary-charging emissions (BEL only; 0 elsewhere).
    .join(stationary_econ.select(*CASE_KEYS, "stat_ghg_tonne"),
          on=CASE_KEYS, how="left")
    .with_columns((pl.col("ghg_tonne") + pl.col("stat_ghg_tonne").fill_null(0.0))
                  .alias("ghg_tonne"))
    .join(freight, on=CASE_KEYS, how="inner")
    # kg CO2 / tonne-km = (tonne * 1000) / (Mt-km * 1e6)
    .with_columns((pl.col("ghg_tonne") / (pl.col("freight_mtkm") * 1000.0))
                  .alias("kg_co2_per_tkm"))
    .pipe(add_case_labels)
)


def cost_grid_facet(chart):
    return chart.facet(
        row=alt.Row("infra_level:N", title="Infrastructure cost scenario",
                    sort=LEVEL_ORDER, header=alt.Header(labelExpr=LEVEL_LABEL_EXPR)),
        column=alt.Column("energy_level:N", title="Energy price differential scenario",
                          sort=LEVEL_ORDER, header=alt.Header(labelExpr=LEVEL_LABEL_EXPR)),
    )


_emis_scale, _emis_domain = tech_option_scale(emissions)
emis_chart = cost_grid_facet(
    alt.Chart(emissions).mark_bar().encode(
        y=alt.Y("tech_option:N", title=None, sort=_emis_domain),
        x=alt.X("kg_co2_per_tkm:Q", title="kg CO2-eq / tonne-km"),
        color=alt.Color("tech_option:N", title="Technology · architecture",
                        scale=_emis_scale, sort=_emis_domain),
        tooltip=["tech_option",
                 alt.Tooltip("kg_co2_per_tkm:Q", format=".5f")],
    ).properties(width=170, height=140)

).properties(title="Lifecycle GHG intensity by technology and cost scenario")

emis_chart

save_svg(emis_chart, "02_emissions_kgco2_per_tkm")

## Figure 3 â€” Lifecycle energy intensity (kWh-eq / tonne-km), stacked by source

In [ ]:
SOURCE_SUBSETS = {
    "Liquid (diesel)": ["Diesel"],
    "Catenary": ["Electricity", "Electricity_Line_Loss"],
    "Stationary charging": ["Electricity_Stationary_Charging"],
}
SOURCE_ORDER = list(SOURCE_SUBSETS)
subset_to_source = {s: k for k, subs in SOURCE_SUBSETS.items() for s in subs}

energy = (
    pl.concat([
        # Catenary + diesel from the sweep metrics (its own stationary subset is 0).
        # Diesel /PLAN_DAYS corrects the Rust over-annualization; electricity is fine.
        (metrics.filter((pl.col("Metric") == "Energy")
                        & pl.col("Subset").is_in(list(subset_to_source))
                        & (pl.col("Subset") != "Electricity_Stationary_Charging"))
         .with_columns(pl.when(pl.col("Subset") == "Diesel")
                         .then(pl.col("Value") / PLAN_DAYS)
                         .otherwise(pl.col("Value")).alias("Value"))
         .with_columns(pl.col("Subset").replace(subset_to_source).alias("source"))
         .group_by(CASE_KEYS + ["source"])
         .agg((pl.col("Value").sum() / MJ_PER_KWH).alias("kwh"))),
        # Reconstructed stationary charging (BEL only).
        stationary_econ.select(
            *CASE_KEYS, pl.lit("Stationary charging").alias("source"),
            (pl.col("stat_energy_mj_total") / MJ_PER_KWH).alias("kwh")),
    ], how="diagonal_relaxed")
    .join(freight, on=CASE_KEYS, how="inner")
    .with_columns((pl.col("kwh") / (pl.col("freight_mtkm") * 1e6)).alias("kwh_per_tkm"))
    .pipe(add_case_labels)
)

# Shared technology ordering (BEL -> Dual-Mode -> Diesel) puts Diesel last = bottom row.
_, _energy_domain = tech_option_scale(energy)
energy_chart = cost_grid_facet(
    alt.Chart(energy).mark_bar().encode(
        y=alt.Y("tech_option:N", title=None, sort=_energy_domain),
        x=alt.X("kwh_per_tkm:Q", title="kWh-eq / tonne-km", stack="zero"),
        color=alt.Color("source:N", title="Energy source", sort=SOURCE_ORDER),
        order=alt.Order("source:N"),
        tooltip=["tech_option", "source",
                 alt.Tooltip("kwh_per_tkm:Q", format=".5f")],
    ).properties(width=170, height=140)
)

save_svg(energy_chart, "03_energy_kwh_per_tkm")
energy_chart


## Figure 4 â€” Lifetime CO2 vs. levelized cost (LCOTKM), scatter by cost scenario


In [ ]:
# Reported LCOTKM has diesel cost over-annualized (numerator) AND freight over-
# annualized (denominator), both by PLAN_DAYS. Exact correction (residual cancels):
#   LCOTKM_corr = PLAN_DAYS*LCOTKM_rep - (PLAN_DAYS-1)*(diesel_cost_disc / tkm_disc)
# using the REPORTED discounted diesel cost and discounted freight per case.
_disc = (metrics.filter(pl.col("Metric") == "Discounting_Factor")
         .select(*CASE_KEYS, "Year", pl.col("Value").alias("disc_factor")))
_diesel_cost_disc = (
    metrics.filter(pl.col("Metric") == "Cost_Diesel")
    .select(*CASE_KEYS, "Year", pl.col("Value").alias("cost_diesel"))
    .join(_disc, on=CASE_KEYS + ["Year"], how="inner")
    .with_columns((pl.col("cost_diesel") / pl.col("disc_factor")).alias("d"))
    .group_by(CASE_KEYS).agg(pl.col("d").sum().alias("diesel_cost_disc"))
)
_tkm_disc = (metrics.filter(pl.col("Metric") == "Mt-km_Discounted")
             .group_by(CASE_KEYS).agg(pl.col("Value").sum().alias("tkm_disc")))

lcotkm = (
    metrics.filter(pl.col("Metric") == "LCOTKM")
    .group_by(CASE_KEYS).agg(pl.col("Value").mean().alias("lcotkm_rep"))
    .join(_diesel_cost_disc, on=CASE_KEYS, how="left")
    .join(_tkm_disc, on=CASE_KEYS, how="left")
    .with_columns(
        (PLAN_DAYS * pl.col("lcotkm_rep")
         - (PLAN_DAYS - 1.0) * pl.col("diesel_cost_disc").fill_null(0.0)
         / pl.col("tkm_disc")).alias("lcotkm_usd_per_mtkm"))
    # Add discounted stationary cost (delta already uses the corrected freight basis).
    .join(stationary_econ.select(*CASE_KEYS, "delta_lcotkm"), on=CASE_KEYS, how="left")
    .with_columns((pl.col("lcotkm_usd_per_mtkm") + pl.col("delta_lcotkm").fill_null(0.0))
                  .alias("lcotkm_usd_per_mtkm"))
    .select(*CASE_KEYS, "lcotkm_usd_per_mtkm")
)

co2_total = (
    # Rebuild from components with diesel put on the week basis (/PLAN_DAYS).
    metrics.filter(pl.col("Metric").is_in(["GHG_Diesel", "GHG_Electricity"]))
    .with_columns(pl.when(pl.col("Metric") == "GHG_Diesel")
                    .then(pl.col("Value") / PLAN_DAYS)
                    .otherwise(pl.col("Value")).alias("Value"))
    .group_by(CASE_KEYS).agg(pl.col("Value").sum().alias("co2_tonne"))
    .join(stationary_econ.select(*CASE_KEYS, "stat_ghg_tonne"), on=CASE_KEYS, how="left")
    .with_columns((pl.col("co2_tonne") + pl.col("stat_ghg_tonne").fill_null(0.0))
                  .alias("co2_tonne"))
    .drop("stat_ghg_tonne")
)

cost_vs_co2 = (
    lcotkm.join(co2_total, on=CASE_KEYS, how="inner")
    .pipe(add_case_labels)
)

_cost_scale, _cost_domain = tech_option_scale(cost_vs_co2)
cost_co2_chart = cost_grid_facet(
    alt.Chart(cost_vs_co2).mark_point(filled=True, size=90, fillOpacity=0.5, stroke="black", strokeWidth=0.6, strokeOpacity=0.7).encode(
        x=alt.X("lcotkm_usd_per_mtkm:Q", title=["Levelized costs", "(USD / million tonne-km)"]),
        y=alt.Y("co2_tonne:Q", title=["Systemwide lifetime", "use-phase tonnes CO2-eq"]),
        color=alt.Color("tech_option:N", title="Technology · architecture",
                        scale=_cost_scale, sort=_cost_domain),
        shape=alt.Shape("tech_family:N", title="Technology", sort=TECH_ORDER),
        tooltip=["tech_option",
                 alt.Tooltip("lcotkm_usd_per_mtkm:Q", format=".1f"),
                 alt.Tooltip("co2_tonne:Q", format=",.0f")],
    ).properties(width=170, height=140)
)
save_svg(cost_co2_chart, "04_lcotkm_vs_co2")
cost_co2_chart


## Figure 5 â€” Optimal technology per cost scenario (3Ã—3 grid)

For each cost scenario (infrastructure cost Ã— energy price) the cell is colored by
the technology that minimizes the chosen metric, and labeled with that technology and
its catenary architecture (architecture is omitted for diesel). The two metric panels
â€” cost-minimizing (LCOTKM) and emissions-minimizing (CO2-eq / tonne-km) â€” sit side by
side, each with one sub-panel per demand level (e.g. regular / high).


In [ ]:
# Demand-level display labels / order for the summary-grid panels.
DEMAND_LABELS = {"central": "Regular demand", "high": "High demand"}
DEMAND_ORDER = ["central", "high"]


def optimal_grid(df, value_col):
    # One row per (demand, infra, energy): the technology minimizing value_col.
    best = (
        df.sort(value_col)
        .group_by(["demand_level", "infra_level", "energy_level"], maintain_order=True)
        .first()
    )
    return best.with_columns(
        pl.col("demand_level").replace(DEMAND_LABELS).alias("demand_display"),
        pl.when(pl.col("tech_family") == "Diesel").then(pl.lit(""))
            .otherwise(pl.col("architecture")).alias("arch_label"),
        pl.col(value_col).alias("metric_value"),
    )


def summary_grid_chart(df, value_col, title, value_fmt, show_legend=True):
    best = optimal_grid(df, value_col)
    present = [d for d in DEMAND_ORDER if d in best["demand_level"].unique().to_list()]
    present += [d for d in best["demand_level"].unique().to_list() if d not in present]
    display_order = [DEMAND_LABELS.get(d, d) for d in present]

    base = alt.Chart(best).encode(
        x=alt.X("energy_level:N", title="Energy price differential scenario",
                sort=LEVEL_ORDER,
                axis=alt.Axis(orient="top", labelAngle=0, labelExpr=LEVEL_LABEL_EXPR)),
        y=alt.Y("infra_level:N", title="Infrastructure cost scenario", sort=LEVEL_ORDER,
                axis=alt.Axis(labelExpr=LEVEL_LABEL_EXPR)),
    )
    rect = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color("tech_family:N", scale=tech_family_scale(), sort=TECH_ORDER,
                        legend=alt.Legend(title="Optimal technology") if show_legend
                        else None),
        tooltip=[alt.Tooltip("tech_option:N", title="Optimal option"),
                 alt.Tooltip("infra_level:N", title="Infra cost"),
                 alt.Tooltip("energy_level:N", title="Energy price"),
                 alt.Tooltip("metric_value:Q", title=value_col, format=value_fmt)],
    )
    tech_text = base.mark_text(fontWeight="bold", fontSize=12, dy=-6,
                               color="white").encode(text="tech_family:N")
    arch_text = base.mark_text(fontSize=10, dy=9, color="white").encode(
        text="arch_label:N")

    return (
        (rect + tech_text + arch_text)
        .properties(width=190, height=190)
        .facet(column=alt.Column("demand_display:N", title=None, sort=display_order))
        .properties(title=title)
    )


# LCOTKM and CO2-eq side by side as panels of a single figure.
lcotkm_grid_chart = summary_grid_chart(
    cost_vs_co2, "lcotkm_usd_per_mtkm",
    "Cost-minimizing (LCOTKM)", ".1f", show_legend=False)
co2_grid_chart = summary_grid_chart(
    emissions, "kg_co2_per_tkm",
    "Emissions-minimizing (CO2-eq / tonne-km)", ".5f", show_legend=True)

optimal_grid_chart = (
    alt.hconcat(lcotkm_grid_chart, co2_grid_chart, spacing=40)
    .properties(title="Optimal technology by cost scenario")
)

save_svg(optimal_grid_chart, "05_optimal_tech_grid")

optimal_grid_chart

## Figure 1 â€” Corridor charging load curves

In [ ]:
def catenary_loss_fraction(case):
    # Line-loss fraction of catenary grid draw for one case, from metrics.
    sub = metrics.filter(
        (pl.col("Metric") == "Energy")
        & (pl.col("architecture") == case["architecture"])
        & (pl.col("variant") == case["variant"])
        & (pl.col("infra_level") == case["infra_level"])
        & (pl.col("energy_level") == case["energy_level"])
        & (pl.col("demand_level") == case["demand_level"])
    )
    deliv = sub.filter(pl.col("Subset") == "Electricity")["Value"].sum() or 0.0
    loss = sub.filter(pl.col("Subset") == "Electricity_Line_Loss")["Value"].sum() or 0.0
    tot = deliv + loss
    return (loss / tot) if tot else 0.0


def _time_columns(df, arch):
    return df.with_columns(
        pl.lit(arch).alias("architecture"),
        (pl.col("time_seconds") / 3600.0).alias("hour"),
        (pl.col("time_seconds") // 86400).cast(pl.Int64).alias("day_index"),
        ((pl.col("time_seconds") / 3600.0) % 24).alias("hour_of_day"),
        (pl.col("power_w") / 1e6).alias("power_mw"),
    )


def load_curve_long(reg_row, per_substation=None):
    # Long-format load curve for one run: time / charging_type / component / power.
    folder = Path(reg_row["folder"])
    arch = reg_row["architecture"]
    frac = catenary_loss_fraction(reg_row)
    frames = []

    lc_path = folder / "substation_load_curves.parquet"
    if lc_path.exists():
        lc = pl.read_parquet(lc_path)
        if per_substation is not None:
            lc = lc.filter(pl.col("substation_id") == per_substation)
        if lc.height:
            tot = (lc.group_by("time_seconds")
                     .agg(pl.col("power_watts").sum().alias("w")).sort("time_seconds"))
            cat = (tot.with_columns(pl.lit("Catenary").alias("charging_type"))
                      .with_columns((pl.col("w") * (1 - frac)).alias("Delivered"),
                                    (pl.col("w") * frac).alias("Losses"))
                      .unpivot(index=["time_seconds", "charging_type"],
                               on=["Delivered", "Losses"],
                               variable_name="component", value_name="power_w"))
            frames.append(cat.select("time_seconds", "charging_type", "component", "power_w"))

    # Stationary (terminal) charging: prefer the run's parquet, else the
    # reconstructed BEL sessions (stationary_sessions) matched to this case.
    if per_substation is None:
        st_path = folder / "stationary_charging.parquet"
        if st_path.exists() and pl.read_parquet(st_path).height:
            st = pl.read_parquet(st_path)
            starts = (st["start_hr"] * 3600).to_list()
            ends = (st["end_hr"] * 3600).to_list()
            kw = st["peak_kw"].to_list()
        elif stationary_sessions.height:
            st = stationary_sessions.filter(
                pl.all_horizontal([pl.col(k) == reg_row[k] for k in CASE_KEYS]))
            starts = (st["start_hr"] * 3600).to_list()
            ends = (st["end_hr"] * 3600).to_list()
            kw = st["grid_kw"].to_list()
        else:
            starts, ends, kw = [], [], []

        if starts:
            base_grid = (frames[0].select("time_seconds").unique().sort("time_seconds")
                         ["time_seconds"].to_list()) if frames else []
            # Include exact session edges so the rectangles render under step-after.
            grid = sorted(set(base_grid) | set(starts) | set(ends))
            # Grid-side draw at each time (float() keeps Float64 when sum -> 0).
            vals = [float(sum(k * 1000.0 for s, e, k in zip(starts, ends, kw) if s <= t < e))
                    for t in grid]
            # Split grid draw into battery-delivered vs charger losses (BEL_CHARGER_EFF).
            # Keep the zero points: under step-after they step the area back down to
            # zero during idle gaps (dropping them bridges the gaps, painting
            # stationary charging as continuous to the end of the run).
            st_df = (pl.DataFrame({
                        "time_seconds": grid,
                        "Delivered": [v * BEL_CHARGER_EFF for v in vals],
                        "Losses": [v * (1.0 - BEL_CHARGER_EFF) for v in vals]})
                     .unpivot(index="time_seconds", on=["Delivered", "Losses"],
                              variable_name="component", value_name="power_w")
                     .with_columns(pl.lit("Stationary").alias("charging_type")))
            frames.append(st_df.select("time_seconds", "charging_type", "component", "power_w"))

    if not frames:
        return pl.DataFrame(schema={"time_seconds": pl.Float64, "charging_type": pl.Utf8,
                                    "component": pl.Utf8, "power_w": pl.Float64,
                                    "architecture": pl.Utf8, "hour": pl.Float64,
                                    "day_index": pl.Int64, "hour_of_day": pl.Float64,
                                    "power_mw": pl.Float64})
    return _time_columns(pl.concat(frames), arch)


def corridor_load_df(variant, infra, energy, demand, per_substation=None):
    sel = registry.filter(
        (pl.col("variant") == variant) & (pl.col("infra_level") == infra)
        & (pl.col("energy_level") == energy) & (pl.col("demand_level") == demand)
    )
    parts = [load_curve_long(r, per_substation) for r in sel.to_dicts()]
    parts = [p for p in parts if p.height]
    if not parts:
        raise ValueError("no load-curve data for the selected scenario")
    return pl.concat(parts)

In [ ]:
# Fully-stacked palette: catenary vs stationary are distinct hues; within each,
# delivered is the dark shade and losses the light shade.
SERIES_ORDER = ["Catenary · Delivered", "Catenary · Losses",
                "Stationary · Delivered", "Stationary · Losses"]
SERIES_COLORS = ["#1f77b4", "#9ecae1", "#e6550d", "#fdae6b"]
CHARGING_TYPE_ORDER = ["Catenary", "Stationary"]
CHARGING_TYPE_COLORS = ["#1f77b4", "#e6550d"]
# Stacking order (bottom -> top): stationary charging sits at the baseline.
SERIES_STACK_ORDER = ["Stationary · Delivered", "Stationary · Losses",
                      "Catenary · Delivered", "Catenary · Losses"]
CHARGING_TYPE_STACK_ORDER = ["Stationary", "Catenary"]


def loadcurve_chart(df, single_day, broken_out, title):
    d = df
    if single_day:
        d = d.filter(pl.col("day_index") == LOADCURVE_DAY_INDEX)
        x = alt.X("hour_of_day:Q", title="Hour of day", scale=alt.Scale(domain=[0, 24]))
    else:
        x = alt.X("hour:Q", title="Hour of simulation")

    y = alt.Y("power_mw:Q", title="Grid power (MW)", stack="zero", aggregate="sum")

    if broken_out:
        _series = pl.col("charging_type") + pl.lit(" · ") + pl.col("component")
        d = d.with_columns(
            _series.alias("series"),
            _series.replace_strict({s: i for i, s in enumerate(SERIES_STACK_ORDER)},
                                   return_dtype=pl.Int64).alias("stack_order"))
        color = alt.Color("series:N", title="Charging type · component",
                          scale=alt.Scale(domain=SERIES_ORDER, range=SERIES_COLORS),
                          sort=SERIES_ORDER)
    else:
        d = d.with_columns(
            pl.col("charging_type").replace_strict(
                {s: i for i, s in enumerate(CHARGING_TYPE_STACK_ORDER)},
                return_dtype=pl.Int64).alias("stack_order"))
        color = alt.Color("charging_type:N", title="Charging type",
                          scale=alt.Scale(domain=CHARGING_TYPE_ORDER,
                                          range=CHARGING_TYPE_COLORS),
                          sort=CHARGING_TYPE_ORDER)

    return (alt.Chart(d).mark_area(interpolate="step-after", opacity=0.9)
            .encode(x=x, y=y, color=color,
                    order=alt.Order("stack_order:Q"),
                    row=alt.Row("architecture:N", title="Architecture"))
            .properties(width=340, height=140, title=title))


corridor = corridor_load_df(LOADCURVE_VARIANT, LOADCURVE_INFRA,
                            LOADCURVE_ENERGY, LOADCURVE_DEMAND)

save_svg(loadcurve_chart(corridor, False, True,
         "Corridor charging load â€” full week, by charging type"),
         "01a_loadcurve_week_by_type")
save_svg(loadcurve_chart(corridor, True, True,
         "Corridor charging load â€” single day, by charging type"),
         "01b_loadcurve_day_by_type")
save_svg(loadcurve_chart(corridor, False, False,
         "Corridor charging load â€” full week, combined"),
         "01c_loadcurve_week_combined")
save_svg(loadcurve_chart(corridor, True, False,
         "Corridor charging load â€” single day, combined"),
         "01d_loadcurve_day_combined")
loadcurve_chart(corridor, False, True, "Corridor charging load â€” full week, by charging type")


## Figure 1a â€” Single mid-route substation load curve (single day)

In [ ]:
def first_midroute_substation(variant, infra, energy, demand):
    sel = registry.filter(
        (pl.col("variant") == variant) & (pl.col("infra_level") == infra)
        & (pl.col("energy_level") == energy) & (pl.col("demand_level") == demand)
    ).to_dicts()
    for r in sel:
        p = Path(r["folder"]) / "substation_load_curves.parquet"
        if p.exists():
            ids = pl.read_parquet(p)["substation_id"].unique().to_list()
            mids = sorted(i for i in ids if i not in ("west_terminus", "east_terminus"))
            if mids:
                return mids[0]
    return None


sub_id = first_midroute_substation(LOADCURVE_VARIANT, LOADCURVE_INFRA,
                                   LOADCURVE_ENERGY, LOADCURVE_DEMAND)
print("mid-route substation:", sub_id)

sub_df = corridor_load_df(LOADCURVE_VARIANT, LOADCURVE_INFRA,
                          LOADCURVE_ENERGY, LOADCURVE_DEMAND, per_substation=sub_id)

sub_chart = save_svg(
    loadcurve_chart(sub_df, True, True,
                    f"Substation {sub_id} charging load \u2014 single day"),
    "01e_substation_day")
sub_chart


In [ ]:
# Two-panel, ac_25kv BEL: whole corridor (full week) vs one mid-route substation (single day).
PANEL_ARCH = "ac_25kv"
_SEP = SERIES_ORDER[0].replace("Catenary", "").replace("Delivered", "")  # exact stored separator


def _panel(df, single_day, title):
    d = df.filter(pl.col("architecture") == PANEL_ARCH)
    if single_day:
        d = d.filter(pl.col("day_index") == LOADCURVE_DAY_INDEX)
        x = alt.X("hour_of_day:Q", title="Hour of day", scale=alt.Scale(domain=[0, 24]))
    else:
        x = alt.X("hour:Q", title="Hour of week")
    series = pl.col("charging_type") + pl.lit(_SEP) + pl.col("component")
    d = d.with_columns(
        series.alias("series"),
        series.replace_strict({s: i for i, s in enumerate(SERIES_STACK_ORDER)},
                              return_dtype=pl.Int64).alias("stack_order"))
    return (alt.Chart(d).mark_area(interpolate="step-after", opacity=0.9).encode(
        x=x,
        y=alt.Y("power_mw:Q", title="Grid power (MW)", stack="zero", aggregate="sum"),
        color=alt.Color("series:N", title="Charging type" + _SEP + "component",
                        scale=alt.Scale(domain=SERIES_ORDER, range=SERIES_COLORS),
                        sort=SERIES_ORDER),
        order=alt.Order("stack_order:Q"),
    ).properties(width=340, height=140, title=title))


# Mid-route substation from the ac_25kv run.
_reg = registry.filter(
    (pl.col("variant") == LOADCURVE_VARIANT) & (pl.col("architecture") == PANEL_ARCH)
    & (pl.col("infra_level") == LOADCURVE_INFRA)
    & (pl.col("energy_level") == LOADCURVE_ENERGY)
    & (pl.col("demand_level") == LOADCURVE_DEMAND)).to_dicts()
panel_sub_id = None
for _r in _reg:
    _p = Path(_r["folder"]) / "substation_load_curves.parquet"
    if _p.exists():
        _ids = pl.read_parquet(_p)["substation_id"].unique().to_list()
        _mids = sorted(i for i in _ids if i not in ("west_terminus", "east_terminus"))
        if _mids:
            panel_sub_id = _mids[0]
            break

sub_panel_df = corridor_load_df(LOADCURVE_VARIANT, LOADCURVE_INFRA, LOADCURVE_ENERGY,
                                LOADCURVE_DEMAND, per_substation=panel_sub_id)

corridor_vs_sub_chart = (
    alt.hconcat(
        _panel(corridor, False, "Full corridor \u2014 full week"),
        _panel(sub_panel_df, True, f"Illustrative substation \u2014 single day"),
        spacing=40,
    )
    .resolve_scale(y="independent")
)
save_svg(corridor_vs_sub_chart, "01f_corridor_vs_substation_ac25kv")
corridor_vs_sub_chart

## Figure 6 â€” Corridor catenary buildout (grade + cumulative energy)

Low infrastructure cost + medium price differential case (`ac_25kv`) shown for the same loaded train in dual-mode and BEL.

In [ ]:
def _shading_layer(shading_pd):
    # fills full chart height by encoding only x/x2
    return (alt.Chart(shading_pd).mark_rect(opacity=0.35, color="darkorange")
            .encode(x=alt.X("x1:Q"), x2="x2:Q"))


def _grade_panel(df, shading_pd, title):
    line = (alt.Chart(df).mark_line(color="#333", strokeWidth=1).encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("elev_m:Q", title="Elevation (m)")))
    return (_shading_layer(shading_pd) + line).properties(width=320, height=130, title=title)


def _energy_panel_dm(df, shading_pd, title):
    df_long = (df[["km", "cat_kwh", "diesel_kwh"]]
               .melt(id_vars="km", var_name="source", value_name="kwh")
               .replace({"cat_kwh": "Electricity", "diesel_kwh": "Diesel"}))
    lines = (alt.Chart(df_long).mark_line().encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("kwh:Q", title="Cumulative energy (kWh-eq)"),
        color=alt.Color("source:N",
                        scale=alt.Scale(domain=["Electricity", "Diesel"],
                                        range=["steelblue", "#ff7f0e"]),
                        legend=alt.Legend(title="Source", orient="top-right"))))
    return (_shading_layer(shading_pd) + lines).properties(width=320, height=130, title=title)


def _energy_panel_bel(df, shading_pd, title, term_kwh):
    line = (alt.Chart(df).mark_line(color="steelblue", strokeWidth=1.5).encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("net_whl_kwh:Q", title="Cumulative energy (kWh-eq)")))
    layers = [_shading_layer(shading_pd), line]
    if term_kwh > 0:
        end_km  = float(df["km"].iloc[-1])
        end_kwh = float(df["net_whl_kwh"].iloc[-1])
        # Vertical segment at final position: stationary terminal recharge.
        seg = (alt.Chart(pd.DataFrame({"km": [end_km, end_km],
                                       "kwh": [end_kwh, end_kwh + term_kwh]}))
               .mark_line(color="steelblue", strokeWidth=2)
               .encode(x="km:Q", y="kwh:Q"))
        layers.append(seg)
    return alt.layer(*layers).properties(width=320, height=130, title=title)


_dm_shading = _cat_shading(_dm_df)
_bel_shading = _cat_shading(_bel_df)

_dm_grade   = _grade_panel(_dm_df,  _dm_shading,  "Dual-Mode · ac_25kv — elevation profile")
_bel_grade  = _grade_panel(_bel_df, _bel_shading, "BEL · ac_25kv — elevation profile")
_dm_energy  = _energy_panel_dm(_dm_df,  _dm_shading,  "Dual-Mode — cumulative energy by source")
_bel_energy = _energy_panel_bel(_bel_df, _bel_shading,
                                "BEL — cumulative net energy + terminal charge",
                                _bel_term_kwh)

buildout_chart = (
    alt.vconcat(
        alt.hconcat(_dm_grade,  _bel_grade,  spacing=30),
        alt.hconcat(_dm_energy, _bel_energy, spacing=30),
        spacing=15,
    ).properties(title="Corridor catenary buildout — ac_25kv, low infra cost, central energy price")
)

save_svg(buildout_chart, "06_corridor_buildout")
buildout_chart

In [ ]:
def _shading_layer(shading_pd):
    # fills full chart height by encoding only x/x2
    return (alt.Chart(shading_pd).mark_rect(opacity=0.35, color="darkorange")
            .encode(x=alt.X("x1:Q"), x2="x2:Q"))


def _grade_panel(df, shading_pd, title):
    line = (alt.Chart(df).mark_line(color="#333", strokeWidth=1).encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("elev_m:Q", title="Elevation (m)")))
    return (_shading_layer(shading_pd) + line).properties(width=320, height=130, title=title)


def _energy_panel_dm(df, shading_pd, title):
    df_long = (df[["km", "cat_kwh", "diesel_kwh"]]
               .melt(id_vars="km", var_name="source", value_name="kwh")
               .replace({"cat_kwh": "Electricity", "diesel_kwh": "Diesel"}))
    lines = (alt.Chart(df_long).mark_line().encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("kwh:Q", title="Cumulative energy (kWh-eq)"),
        color=alt.Color("source:N",
                        scale=alt.Scale(domain=["Electricity", "Diesel"],
                                        range=["steelblue", "#ff7f0e"]),
                        legend=alt.Legend(title="Source", orient="top-left"))))
    return (_shading_layer(shading_pd) + lines).properties(width=320, height=130, title=title)


def _energy_panel_bel(df, shading_pd, title, term_kwh):
    line = (alt.Chart(df).mark_line(color="steelblue", strokeWidth=1.5).encode(
        x=alt.X("km:Q", title="Distance (km)"),
        y=alt.Y("net_whl_kwh:Q", title="Cumulative energy (kWh-eq)")))
    layers = [_shading_layer(shading_pd), line]
    if term_kwh > 0:
        end_km  = float(df["km"].iloc[-1])
        end_kwh = float(df["net_whl_kwh"].iloc[-1])
        # Vertical segment at final position: stationary terminal recharge.
        seg = (alt.Chart(pd.DataFrame({"km": [end_km, end_km],
                                       "kwh": [end_kwh, end_kwh + term_kwh]}))
               .mark_line(color="steelblue", strokeWidth=2)
               .encode(x="km:Q", y="kwh:Q"))
        layers.append(seg)
    return alt.layer(*layers).properties(width=320, height=130, title=title)


_dm_shading = _cat_shading(_dm_df)
_bel_shading = _cat_shading(_bel_df)

_dm_grade   = _grade_panel(_dm_df,  _dm_shading,  "Dual-Mode · ac_25kv — elevation profile")
_bel_grade  = _grade_panel(_bel_df, _bel_shading, "BEL · ac_25kv — elevation profile")
_dm_energy  = _energy_panel_dm(_dm_df,  _dm_shading,  "Dual-Mode — cumulative energy by source")
_bel_energy = _energy_panel_bel(_bel_df, _bel_shading,
                                "BEL — cumulative net energy + terminal charge",
                                _bel_term_kwh)

buildout_chart = (
    alt.vconcat(
        alt.hconcat(_dm_grade,  _bel_grade,  spacing=30).resolve_legend(color="independent"),
        alt.hconcat(_dm_energy, _bel_energy, spacing=30).resolve_legend(color="independent"),
        spacing=15,
    )
    .resolve_legend(color="independent")
    #.properties(title="Corridor catenary buildout — ac_25kv, low infra cost, central energy price")
)

save_svg(buildout_chart, "06_corridor_buildout")
buildout_chart

## Table — Corridor results across cost scenarios (LaTeX, booktabs)

First-level columns are powertrains (diesel excluded), second-level columns catenary
architectures. Each variable is reported as three sub-rows across the nine
infrastructure-cost × energy-price cost cells (`demand = central`):

- **Low / High** — the min / max of the variable across the nine cost cells.
- **Central** — the reference cell (medium infrastructure cost, medium energy price).

Variables & units:
- **Substations built** — the model's `num_substations` (built substation count).
- **Diesel fuel reduction (%)** — `1 − case_diesel / all-diesel_baseline` (the Rust
  over-annualization cancels in the ratio).
- **Median substation peak demand (MW)** / **energy (weekly MWh)** — per-substation
  peak / time-integrated energy (60 s bins) from `substation_load_curves.parquet`,
  medianed over the substations carrying catenary load; a cell that electrifies
  nothing contributes 0.

BEL uses one reference buildout, so its three rows coincide; the dual-mode buildout is
re-optimized per cost cell, so many cells electrify little or nothing. Requires the
`booktabs` and `multirow` LaTeX packages.


In [ ]:
# --- Scenario results table (LaTeX, booktabs) ------------------------------------
# Physical quantities per case across the nine infra x energy cost cells. Each
# variable gets three labeled sub-rows: Low / High = min / max across the nine cells,
# Central = the reference (infra=med, energy=med) cell. Units are chosen per variable
# to keep the columns narrow.
SUBSTATION_BIN_S = 60.0  # SIM_SAVE_INTERVAL used when the sweep binned load curves

ARCH_ORDER = ["ac_25kv", "ac_2x25", "dc_9kv"]
ARCH_LABEL = {"ac_25kv": r"AC 25\,kV",
              "ac_2x25": r"AC 2$\times$25\,kV",
              "dc_9kv": r"DC 9\,kV"}
PT_ORDER = ["dual_mode", "bel"]
PT_LABEL = {"bel": "BEL", "dual_mode": "Dual-mode"}
CENTRAL_INFRA, CENTRAL_ENERGY = "med", "med"

# Diesel energy per case; reduction vs the all-diesel baseline (annualization cancels).
_diesel_e = (metrics.filter((pl.col("Metric") == "Energy") & (pl.col("Subset") == "Diesel"))
             .group_by(CASE_KEYS).agg(pl.col("Value").sum().alias("diesel_mj")))
_baseline_diesel_mj = _diesel_e.filter(pl.col("variant") == "diesel")["diesel_mj"].max()

# num_substations rides on every metric row (constant within a case).
_nsub = (metrics.with_columns(pl.col("num_substations").cast(pl.Int64, strict=False))
         .group_by(CASE_KEYS).agg(pl.col("num_substations").max().alias("n_substations")))


def _substation_medians(folder):
    # Median (peak kW, weekly kWh) over substations carrying catenary load; a cell
    # that electrifies nothing has an empty curve -> 0 (no loaded substation).
    p = Path(folder) / "substation_load_curves.parquet"
    if not p.exists():
        return 0.0, 0.0
    lc = pl.read_parquet(p)
    if lc.height == 0:
        return 0.0, 0.0
    per = lc.group_by("substation_id").agg(
        (pl.col("power_watts").max() / 1e3).alias("peak_kw"),
        (pl.col("power_watts").sum() * SUBSTATION_BIN_S / 3.6e6).alias("weekly_kwh"))
    return float(per["peak_kw"].median()), float(per["weekly_kwh"].median())


_rows = []
for r in registry.filter(pl.col("variant").is_in(PT_ORDER)).to_dicts():
    pk, wk = _substation_medians(r["folder"])
    _rows.append({**{k: r[k] for k in CASE_KEYS},
                  "median_peak_kw": pk, "median_weekly_kwh": wk})

per_case = (
    pl.DataFrame(_rows)
    .join(_nsub, on=CASE_KEYS, how="left")
    .join(_diesel_e, on=CASE_KEYS, how="left")
    .with_columns(((1.0 - pl.col("diesel_mj") / _baseline_diesel_mj) * 100.0)
                  .alias("diesel_reduction_pct"))
)

# (column, row label incl. unit, unit scale, decimals). Scales keep the numbers narrow.
VARIABLES = [
    ("n_substations",        "Substations built",                     1.0, 0),
    ("diesel_reduction_pct", r"Diesel fuel reduction (\%)",           1.0, 1),
    ("median_peak_kw",       "Median substation peak demand (MW)",    1e-3, 1),
    ("median_weekly_kwh",    "Median substation energy (weekly MWh)", 1e-3, 0),
]
COLS = [c for c, _, _, _ in VARIABLES]

_lohi = per_case.group_by("variant", "architecture").agg(
    [pl.col(c).min().alias(f"{c}__lo") for c in COLS]
    + [pl.col(c).max().alias(f"{c}__hi") for c in COLS])
_LOHI = {(r["variant"], r["architecture"]): r for r in _lohi.to_dicts()}
_CENTRAL = {(r["variant"], r["architecture"]): r for r in per_case.filter(
    (pl.col("infra_level") == CENTRAL_INFRA) & (pl.col("energy_level") == CENTRAL_ENERGY)
).to_dicts()}


def _fmt(x, scale, dec):
    return f"{x * scale:,.{dec}f}".replace(",", r"\,")  # thin-space thousands


# Scenario sub-rows: (label, value getter for a (powertrain, architecture) cell).
SCEN_ROWS = [
    ("Low",     lambda pt, ar, col: _LOHI[(pt, ar)][f"{col}__lo"]),
    ("Central", lambda pt, ar, col: _CENTRAL[(pt, ar)][col]),
    ("High",    lambda pt, ar, col: _LOHI[(pt, ar)][f"{col}__hi"]),
]

_arch_header = " & ".join(ARCH_LABEL[a] for a in ARCH_ORDER)
_lines = [
    r"\begin{table}[t]",
    r"  \centering",
    r"  \footnotesize",
    r"  \caption{Corridor results across the nine infrastructure-cost $\times$ "
    r"energy-price scenarios (central demand). Each variable is reported as three "
    r"rows: \emph{Low} and \emph{High} are the min and max across the nine cost "
    r"cells, and \emph{Central} is the reference (medium infrastructure cost, medium "
    r"energy price) cell. BEL uses a single reference buildout, so its three rows "
    r"coincide; the dual-mode buildout is re-optimized per cost cell, so many cells "
    r"electrify little or nothing (hence the low/central zeros).}",
    r"  \label{tab:corridor-scenario-results}",
    r"  \begin{tabular}{@{}ll cccccc@{}}",
    r"    \toprule",
    r"    & & " + " & ".join(r"\multicolumn{3}{c}{" + PT_LABEL[pt] + r"}"
                             for pt in PT_ORDER) + r" \\",
    r"    \cmidrule(lr){3-5}\cmidrule(lr){6-8}",
    r"    Variable & Scenario & " + _arch_header + r" & " + _arch_header + r" \\",
    r"    \midrule",
]
for i, (col, label, scale, dec) in enumerate(VARIABLES):
    for j, (scen, getval) in enumerate(SCEN_ROWS):
        head = (r"\multirow{3}{*}{" + label + r"}") if j == 0 else ""
        cells = [_fmt(getval(pt, ar, col), scale, dec)
                 for pt in PT_ORDER for ar in ARCH_ORDER]
        _lines.append("    " + head + " & " + scen + " & "
                      + " & ".join(cells) + r" \\")
    _lines.append(r"    \bottomrule" if i == len(VARIABLES) - 1 else r"    \midrule")
_lines += [r"  \end{tabular}", r"\end{table}"]
latex_table = "\n".join(_lines)

_tex_path = FIG_DIR / "07_scenario_results_table.tex"
_tex_path.write_text(latex_table, encoding="utf-8")
print("saved", _tex_path.relative_to(SWEEP_DIR.parent))
print(latex_table)


In [ ]:
import sys
import types

import numpy as np
import matplotlib.pyplot as plt
import altrios
import altrios.energization as en
from IPython.display import Image, display

# Make the sibling plotting module (next to this notebook) importable, then reuse
# the exact routine the sweep uses to render each run's map.pdf.
_APP_DIR = SWEEP_DIR.parents[2]
if str(_APP_DIR) not in sys.path:
    sys.path.insert(0, str(_APP_DIR))
from plot_electrified_network import plot_electrified_track, corridor_projection

# Fixed scenario: low infrastructure cost, medium fuel-price differential (central
# demand). Columns = build variant (dual-mode | BEL); rows = architecture.
MAP_INFRA, MAP_ENERGY = "low", "med"
MAP_ROWS = ["ac_25kv", "ac_2x25", "dc_9kv"]
MAP_COLS = ["dual_mode", "bel"]
MAP_ARCH_LABEL = {"ac_25kv": "AC 25 kV", "ac_2x25": "AC 2\u00d725 kV",
                  "dc_9kv": "DC 9 kV"}
MAP_VARIANT_LABEL = {"dual_mode": "Dual-mode", "bel": "BEL"}

# p95-consist design train power -> per-architecture feed distance, matching the
# substation-spacing model the sweep used (energization_optimization_corridor.py).
_nloc = (pl.read_csv(CONSIST_PLAN_PATH)
         .group_by("Train_ID").agg(pl.len().alias("n")))["n"].to_list()
_DESIGN_P_W = float(np.percentile(_nloc, 95.0)) * 5.0e6
_FEED_MI = {a: en.suggest_optimization_inputs(
                a, design_train_power_watts=_DESIGN_P_W
            ).substation_feed_distance_miles for a in MAP_ROWS}


def _map_folder(arch, variant):
    row = registry.filter(
        (pl.col("architecture") == arch) & (pl.col("variant") == variant)
        & (pl.col("infra_level") == MAP_INFRA)
        & (pl.col("energy_level") == MAP_ENERGY)
        & (pl.col("demand_level") == REPORT_DEMAND))
    if row.height == 0:
        raise FileNotFoundError(f"no run for {arch} / {variant}")
    return Path(row["folder"][0])


def _energized_intervals(folder):
    # optimization.parquet -> {link_idx: [(start_m, end_m), ...]} for plot_electrified_track.
    df = pl.read_parquet(folder / "optimization.parquet")
    out: dict = {}
    for li, a, b in df.select(
            "link_idx", "offset_start_meters", "offset_end_meters").iter_rows():
        if li is None:
            continue
        out.setdefault(int(li), []).append((float(a), float(b)))
    return out


def _substations(folder):
    # substations.parquet -> lightweight objects (.lat/.lon/.id/.existing) + built ids.
    df = pl.read_parquet(folder / "substations.parquet")
    subs = [types.SimpleNamespace(id=r["id"], lat=r["lat"], lon=r["lon"],
                                  existing=r["existing"], link_idx=None)
            for r in df.iter_rows(named=True)]
    built = [r["id"] for r in df.iter_rows(named=True) if r["built"]]
    return subs, built


# One shared network (identical corridor geometry across scenarios) fixes the
# projection so every panel registers to the same map extent.
_net_dir = _map_folder("ac_25kv", "dual_mode")
# Sweeps now write msgpack; fall back to yaml for runs made before the switch.
_net_path = _net_dir / "network.msgpack"
if not _net_path.exists():
    _net_path = _net_dir / "network.yaml"
_net = altrios.Network.from_file(str(_net_path))
_proj = corridor_projection(_net)

# Full text-block page (portrait) minus running heads/feet.
fig, axes = plt.subplots(
    len(MAP_ROWS), len(MAP_COLS), figsize=(7.2, 9.4),
    subplot_kw={"projection": _proj})

for i, arch in enumerate(MAP_ROWS):
    for j, variant in enumerate(MAP_COLS):
        folder = _map_folder(arch, variant)
        subs, built = _substations(folder)
        plot_electrified_track(
            _net, ax=axes[i][j],
            energized_link_intervals=_energized_intervals(folder),
            substations=subs, built_substation_ids=built,
            substation_feed_distance_miles=_FEED_MI[arch],
            title=f"{MAP_ARCH_LABEL[arch]} \u2014 {MAP_VARIANT_LABEL[variant]}",
        )

fig.tight_layout()

_maps_pdf = FIG_DIR / "08_energization_maps_panel.pdf"
_maps_png = FIG_DIR / "08_energization_maps_panel.png"
fig.savefig(str(_maps_pdf), bbox_inches="tight")
fig.savefig(str(_maps_png), dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved", _maps_pdf.relative_to(SWEEP_DIR.parent))
display(Image(filename=str(_maps_png)))